# Bipolar LFP visualization

Fetch the 1–250 Hz LFP band and plot one hippocampal probe.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import spyglass.lfp as lfp
from spyglass.lfp.analysis.v1 import lfp_band
from seizure_detection_ll.utils import bipolar_montage, probe_bad_channels


In [ ]:
rat_name, date, epoch_to_plot = "SC137", "20260218", "11_s6"
nwb_file_name = f"{rat_name}{date}_.nwb"
lfp_electrode_group_name = f"all_hippocampal_probes_all128ch_{rat_name}"
filter_name = "IED filt 1-250hz fs 1khz"
lfp_band_sampling_rate = 1000
merge_id = (lfp.LFPOutput.LFPV1() & {"nwb_file_name": nwb_file_name,
            "lfp_electrode_group_name": lfp_electrode_group_name,
            "target_interval_list_name": epoch_to_plot}).fetch1("merge_id")

# Create LFP band selection and populate it first if needed:
# lfp_band.LFPBandSelection().set_lfp_band_electrodes(
#     nwb_file_name=nwb_file_name, lfp_merge_id=merge_id,
#     electrode_list=list(range(256, 640)), filter_name=filter_name,
#     interval_list_name=epoch_to_plot, reference_electrode_list=[-1],
#     lfp_band_sampling_rate=lfp_band_sampling_rate)
# lfp_band.LFPBandV1().populate(lfp_band.LFPBandSelection() & {
#     "lfp_merge_id": merge_id, "filter_name": filter_name})

lfp_band_key = (lfp_band.LFPBandSelection() & dict(lfp_merge_id=merge_id,
                filter_name=filter_name, lfp_band_sampling_rate=lfp_band_sampling_rate)).fetch1("KEY")
filtered_lfp = (lfp_band.LFPBandV1() & lfp_band_key).fetch1_dataframe()
bad_channels = probe_bad_channels(nwb_file_name)


In [ ]:
probe_idx = 0  # 0=L_hpc1, 1=L_hpc2, 2=R_hpc
lfp_probe = filtered_lfp.iloc[:, probe_idx*128:(probe_idx+1)*128]
bipolar_lfp = bipolar_montage(lfp_probe, bad_channels[probe_idx])


In [ ]:
plot_df = bipolar_lfp.loc[:] 
t = plot_df.index.to_numpy(dtype=float)
x = plot_df.to_numpy(dtype=float)
spacing = 5 * np.nanmedian(np.nanpercentile(np.abs(x), 95, axis=0))
fig, ax = plt.subplots(figsize=(16, 12))
for i, col in enumerate(plot_df.columns):
    ax.plot(t, x[:, i] + i*spacing, linewidth=0.6)
ax.set_yticks(np.arange(x.shape[1])*spacing, labels=[f"{a}: {b}" for a, b in plot_df.columns], fontsize=7)
ax.set(xlabel="Time (s)", ylabel="Bipolar channel", title=f"Bipolar LFP: {['L_hpc1', 'L_hpc2', 'R_hpc'][probe_idx]} ({epoch_to_plot})")
ax.invert_yaxis()
plt.tight_layout()
plt.show()
